# Chart2Data Fine-tuning with Unsloth

Fine-tune a **Qwen3.5** vision language model on the Chart2Data dataset to
extract numerical time-series data from chart images.

Pipeline:
Load dataset → Format as vision chat messages → Fine-tune with LoRA →
Evaluate → Push adapter to HuggingFace → Chat with Unsloth Studio UI.

## Configuration
Change the parameters below before running.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Configuration
# ═══════════════════════════════════════════════════════════════════════════

# Model to fine-tune (Qwen3.5 vision models via Unsloth)
MODEL_ID = "unsloth/Qwen3.5-0.8B"
# Alternatives:
# MODEL_ID = "unsloth/Qwen3.5-2B"   # Needs ~12GB VRAM
# MODEL_ID = "unsloth/Qwen3.5-4B"   # Needs ~24GB VRAM

# Dataset on HuggingFace
DATASET_REPO = "maurovidal/chart2data-v0-sft"
DATASET_SPLIT_TRAIN = "train"
DATASET_SPLIT_VALIDATION = "validation"

# Output model repository on HuggingFace
MODEL_REPO = "maurovidal/chart2data-qwen3.5-0.8b"
MODEL_REVISION = "v0.1"
MODEL_USERNAME = "maurovidal"

# Training hyperparameters
MAX_SEQ_LEN = 2048
EPOCHS = 3
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch = BATCH_SIZE * STEPS
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
WEIGHT_DECAY = 0.001
WARMUP_RATIO = 0.05
EVAL_STEPS = 50
SAVE_STEPS = 50
MAX_EVAL_SAMPLES = 200

print("Configuration loaded.")
print(f"  Model: {MODEL_ID}")
print(f"  Dataset: {DATASET_REPO}")
print(f"  Output repo: {MODEL_REPO} (revision: {MODEL_REVISION})")
print(f"  Epochs: {EPOCHS} | Batch: {BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS}")
print(f"  LoRA: r={LORA_R}, alpha={LORA_ALPHA}")
print(f"  LR: {LEARNING_RATE} | Seq len: {MAX_SEQ_LEN}")

## Step 1: Install dependencies

In [ ]:
!pip install -q unsloth trl datasets peft transformers accelerate bitsandbytes

## Step 2: Import libraries

In [ ]:
import json
import numpy as np
import torch
from pathlib import Path
from datasets import load_dataset
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

## Step 3: HuggingFace login

In [ ]:
from huggingface_hub import login
try:
    login()
    print("✓ HuggingFace logged in")
except Exception as e:
    print(f"Login needed: {e}")
    token = input("Paste your HF token: ")
    login(token=token)

## Step 4: Load and format the dataset

In [ ]:
print(f"Loading dataset: {DATASET_REPO}")
dataset = load_dataset(DATASET_REPO, split="train")
print(f"Train set: {len(dataset):,} samples")
print(f"Columns: {dataset.column_names}")
print()
print("Sample row keys:", list(dataset[0].keys()))
print("User message structure:", dataset[0]['messages'][0])
print()
# Convert the image format for Qwen3.5-VL
def convert_image_format(row):
    """Convert {"type": "image", "text": ""} -> {"type": "image", "image": pil_image}"""
    new_messages = []
    for msg in row['messages']:
        new_content = []
        for part in msg['content']:
            if part['type'] == 'image' and 'image' not in part and 'image' in row:
                new_content.append({'type': 'image', 'image': row['image']})
            else:
                new_content.append(part)
        new_messages.append({'role': msg['role'], 'content': new_content})
    return {'messages': new_messages, 'image': row.get('image')}

dataset = dataset.map(convert_image_format)
print(f"✓ Converted image format for vision model")
print(f"  Sample user content: {dataset[0]['messages'][0]['content'][:2]}")


## Step 5: Load the vision model

In [ ]:
# FastVisionModel handles vision (Qwen3.5-Vision, Qwen2.5-VL, Llama-3.2-Vision, etc.)
print(f"Loading model: {MODEL_ID}")
model, processor = FastVisionModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,  # 16bit LoRA for better quality
    use_gradient_checkpointing="unsloth",  # Memory-efficient checkpointing
)
# The processor wraps the tokenizer; extract it for training
tokenizer = processor.tokenizer
print(f"✓ Model loaded: {model.config.model_type}")
print(f"  Dtype: {model.dtype}")
print(f"  Max sequence length: {MAX_SEQ_LEN}")
print(f"  Vocabulary size: {len(tokenizer)}")


## Step 6: Apply LoRA adapters

In [ ]:
# FastVisionModel.get_peft_model handles vision-specific LoRA parameters
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,     # Fine-tune the vision encoder too
    finetune_language_layers=True,   # Fine-tune the language model too
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    use_rslora=False,
    random_state=3407,
)

model.print_trainable_parameters()

## Step 7: Train the model

In [ ]:
# FastVisionModel.for_training() switches model from inference to training mode
FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),  # REQUIRED for vision
    train_dataset=dataset,
    args=SFTConfig(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        warmup_steps=WARMUP_RATIO,
        max_steps=0,  # 0 = use n_epochs
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="linear",
        logging_steps=10,
        optim="adamw_8bit",
        fp16=True,
        bf16=False,
        report_to="none",
        num_train_epochs=EPOCHS,
        eval_steps=EVAL_STEPS,
        evaluation_strategy="steps",
        save_steps=SAVE_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model="loss",
        save_strategy="steps",
        # REQUIRED for vision:
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=MAX_SEQ_LEN,
    ),
)

In [ ]:
print("Starting training...")
print(f"  Epochs: {EPOCHS}")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  This may take 30-90 minutes.")
print()
trainer.train()

## Step 8: Evaluate on validation set

In [ ]:
from datasets import load_dataset

eval_dataset = load_dataset(DATASET_REPO, split=DATASET_SPLIT_VALIDATION)
print(f"Validation set: {len(eval_dataset):,} samples")

# Sample a subset for faster evaluation
eval_samples = eval_dataset.select(range(min(MAX_EVAL_SAMPLES, len(eval_dataset))))
print(f"Evaluating on {len(eval_samples)} samples...")

### Evaluation helpers

In [ ]:
def eval_json_parse(text: str) -> dict:
    """Parse JSON from model response."""
    try:
        text = text.strip()
        if text.startswith("```"):
            lines = text.split("\n")
            for i, line in enumerate(lines):
                if "json" in line.lower():
                    text = "\n".join(lines[i+1:])
                    break
            text = text.split("```")[0]
        data = json.loads(text)
        if "x" in data and "y" in data:
            return data
    except (json.JSONDecodeError, KeyError):
        pass
    return None

def mae(actual: list, predicted: list) -> float:
    if len(actual) != len(predicted): return float("inf")
    return float(np.mean(np.abs(np.array(actual) - np.array(predicted))))

def rmse(actual: list, predicted: list) -> float:
    if len(actual) != len(predicted): return float("inf")
    return float(np.sqrt(np.mean((np.array(actual) - np.array(predicted))**2)))

def normalized_mae(actual: list, predicted: list) -> float:
    if len(actual) != len(predicted): return float("inf")
    span = max(actual) - min(actual) if max(actual) != min(actual) else 1.0
    return float(np.mean(np.abs(np.array(actual) - np.array(predicted))) / span)

def parse_success_rate(samples: list) -> float:
    if not samples: return 0.0
    parsed = [eval_json_parse(s["assistant_text"]) for s in samples]
    return sum(1 for p in parsed if p is not None) / len(samples)

### Run inference

In [ ]:
# Enable inference mode
FastVisionModel.for_inference(model)

In [ ]:
# The dataset format from unsloth's vision SFT:
# messages[0] = {"role": "user", "content": [{"type": "image", "image": pil_image}, {"type": "text", "text": instruction}]}
# messages[1] = {"role": "assistant", "content": [{"type": "text", "text": "{\"x\": [...], \"y\": [...]}"}]}
#
# But chart2data-v0-sft uses messages[0] = {"role": "user", "content": [{"type": "image", "text": ""}, {"type": "text", "text": "..."}]}
# We need to convert: {"type": "image", "text": ""} → {"type": "image", "image": pil_image}

predictions = []
mae_x_vals, mae_y_vals, rmse_y_vals, nmae_y_vals = [], [], [], []
parse_success = 0
exact_match = 0
total = 0

print("Running inference...")
for i, row in enumerate(eval_samples):
    messages = row["messages"]
    ground_truth = messages[1]["content"][0]["text"]
    user_content = messages[0]["content"]

    # Convert {"type": "image", "text": ""} → {"type": "image", "image": image}
    user_content_vision = []
    image_img = None
    text_content = ""
    for part in user_content:
        if part["type"] == "image" and "image" not in part:
            # This is the chart2data format - add the image
            user_content_vision.append({"type": "image", "image": part.get("image", row["image"])})
            image_img = part.get("image", row["image"])
        else:
            user_content_vision.append(part)
            if part["type"] == "text" and "text" in part:
                text_content += part["text"] + " "

    # Build messages for model
    model_messages = [
        {"role": "user", "content": user_content_vision},
    ]

    # Apply chat template
    input_text = tokenizer.apply_chat_template(
        model_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Get the image (first image in user content)
    img = None
    for part in user_content_vision:
        if part["type"] == "image":
            img = part["image"]
            break

    if img is None:
        continue

    # Tokenize with image
    inputs = tokenizer(
        image=img,
        text=input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to(model.device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            use_cache=True,
            temperature=0.1,
            top_p=0.9,
        )

    # Decode
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract the assistant response
    assistant_text = generated
    if "assistant" in generated:
        assistant_text = generated.split("assistant")[-1].strip()
    if assistant_text.startswith("```"):
        lines = assistant_text.split("\n")
        for j, line in enumerate(lines):
            if "json" in line.lower():
                assistant_text = "\n".join(lines[j+1:])
                break
        assistant_text = assistant_text.split("```")[0].strip()

    # Parse and compare
    predicted = eval_json_parse(assistant_text)
    total += 1

    if predicted:
        try:
            gt = json.loads(ground_truth)
            if "x" in gt and "y" in gt and len(predicted["x"]) == len(gt["x"]) and len(predicted["y"]) == len(gt["y"]):
                m_x = mae(gt["x"], predicted["x"])
                m_y = mae(gt["y"], predicted["y"])
                r_y = rmse(gt["y"], predicted["y"])
                n_m_y = normalized_mae(gt["y"], predicted["y"])
                mae_x_vals.append(m_x)
                mae_y_vals.append(m_y)
                rmse_y_vals.append(r_y)
                nmae_y_vals.append(n_m_y)
                parse_success += 1
                if m_x < 1e-6 and m_y < 1e-6:
                    exact_match += 1
        except Exception:
            pass

    if (i + 1) % 50 == 0:
        print(f"  Processed {i+1}/{len(eval_samples)}")

# Metrics
print("\n" + "="*60)
print("Evaluation Results")
print("="*60)
print(f"  Total samples:              {total}")
print(f"  Valid parses:               {parse_success}")
print(f"  Parse success rate:         {parse_success/total*100:.1f}%" if total else "  N/A")
print(f"  Exact match rate:           {exact_match/total*100:.1f}%" if total else "  N/A")
if mae_x_vals:
    print(f"  MAE (X):                    {np.mean(mae_x_vals):.4f}")
    print(f"  MAE (Y):                    {np.mean(mae_y_vals):.4f}")
    print(f"  RMSE (Y):                   {np.mean(rmse_y_vals):.4f}")
    print(f"  NMAE (Y):                   {np.mean(nmae_y_vals):.4f}")
print("="*60)

## Step 9: Save and push to HuggingFace

### Save LoRA adapter

In [ ]:
adapter_path = f"./lora_{MODEL_ID.split('/')[-1]}_{MODEL_REVISION}"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"✓ Adapter saved to: {adapter_path}")

# Save training config for reproduction
import yaml
config = {
    "model_id": MODEL_ID,
    "dataset_repo": DATASET_REPO,
    "model_repo": MODEL_REPO,
    "revision": MODEL_REVISION,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "max_seq_len": MAX_SEQ_LEN,
}
with open(f"{adapter_path}/training_config.yaml", "w") as f:
    yaml.dump(config, f)
print(f"✓ Training config saved")

### Push to HuggingFace

In [ ]:
from huggingface_hub import create_repo, upload_folder

try:
    create_repo(repo_id=MODEL_REPO, exist_ok=True, repo_type="model")
    print(f"✓ Repo ready: {MODEL_REPO}")
except Exception as e:
    print(f"Repo error: {e}")

upload_folder(
    folder_path=adapter_path,
    repo_id=f"{MODEL_USERNAME}/{MODEL_REPO.split('/')[-1]}",
    revision=MODEL_REVISION,
)
print(f"✓ Pushed to: https://huggingface.co/{MODEL_USERNAME}/{MODEL_REPO.split('/')[-1]}/tree/{MODEL_REVISION}")

### Merge and push full model (optional)

In [ ]:
# Uncomment to merge and upload the full 16-bit model:
# print("Merging adapter (may take a few minutes)...")
# model.save_pretrained_merged(f"{adapter_path}/merged", tokenizer, save_method="merged_16bit")
# upload_folder(
#     folder_path=f"{adapter_path}/merged",
#     repo_id=f"{MODEL_USERNAME}/{MODEL_REPO.split('/')[-1]}",
#     revision=f"{MODEL_REVISION}-merged",
# )
# print(f"✓ Merged model: https://huggingface.co/{MODEL_USERNAME}/{MODEL_REPO.split('/')[-1]}/tree/{MODEL_REVISION}-merged")

## Step 10: Chat with the model (Inference mode)

In [ ]:
FastVisionModel.for_inference(model)

print("="*60)
print("Chat with your fine-tuned model")
print("Type 'exit' to quit.")
print("="*60)
print()

# Load sample images from the dataset for testing
train_ds = load_dataset(DATASET_REPO, split=DATASET_SPLIT_TRAIN)
sample_count = min(10, len(train_ds))

img_idx = 0
while True:
    user_input = input("\nYou: ").strip()
    if user_input.lower() in ["exit", "quit"]:
        print("\nGoodbye! 👋")
        break
    
    # Cycle through sample images
    sample = train_ds[img_idx % sample_count]
    img_idx += 1
    img = sample["image"]
    instruction = user_input
    
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": instruction}
        ]}
    ]
    
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    inputs = tokenizer(
        image=img,
        text=input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            use_cache=True,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    if response.startswith("```"):
        response = response.split("```")[0].strip()
    
    print(f"\nModel: {response}")
    print()

## Summary

You've just:
1. ✅ Loaded the Chart2Data dataset with vision support
2. ✅ Fine-tuned Qwen3.5 with LoRA (vision + language layers)
3. ✅ Evaluated the model on validation data
4. ✅ Saved the adapter and pushed to HuggingFace
5. ✅ Tested the model in interactive chat mode

### Next steps
- **Try larger models:** Change `MODEL_ID` to `unsloth/Qwen3.5-2B` or `unsloth/Qwen3.5-4B` (needs more VRAM)
- **More training:** Increase `EPOCHS` or add more data
- **Merge the model:** Uncomment the merge cell in Step 9
- **Use with Unsloth Studio:** Load the adapter with `FastVisionModel.from_pretrained(MODEL_REPO, revision=MODEL_REVISION)`

### Tips
- **More epochs:** 5-10 can improve results, watch for overfitting
- **Larger model:** 2B or 4B will generally perform better than 0.8B
- **Lower LR:** If unstable, try `1e-4` or `5e-5`
- **Higher LoRA rank:** Try `LORA_R=32` or `LORA_R=64` for more capacity
- **More data:** Use a larger dataset for better generalization